In [ ]:
# colab <-> gpu 연동
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# !pip install langchain_openai langchain_community langchain_chroma pypdf

In [ ]:
# pip install langchain-text-splitters

In [ ]:
import os
import json
import requests
from typing import List

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.documents import Document

In [ ]:
os.environ['OPENAI_API_KEY'] = 'sk-proj-'

In [ ]:
# 분석할 PDF 파일을 웹에서 다운로드.
url = "https://github.com/llama-index-tutorial/llama-index-tutorial/raw/main/ch07/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf"
filename = "2023_북한인권보고서.pdf"

response = requests.get(url)
with open(filename, "wb") as f:
    f.write(response.content)

print(f"{filename} 다운로드 완료")

2023_북한인권보고서.pdf 다운로드 완료


In [ ]:
# ===== LLM & 임베딩 =====
llm = ChatOpenAI(
    model="gpt-4o",   # 속도/비용 안정 (gpt-4o 써도 ok)
    temperature=0.2
)

embed_model = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

# ===== 문서 분할 =====
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=100,
)

# ===== PDF 로드 =====
loader = PyPDFLoader("2023_북한인권보고서.pdf")
documents = loader.load()

# ===== 문서 분할 =====
chunks = text_splitter.split_documents(documents)

# ===== 벡터 DB 생성 =====
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embed_model,
    persist_directory="./chroma_db"   # 선택 (재사용 가능)
)

In [ ]:
class DocumentScorer:
    # LLM을 사용해 문서의 관련성을 정밀하게 평가하고 점수를 매기는 클래스

    def __init__(self, llm):
        self.llm = llm

    def evaluate_document(self, query: str, content: str) -> float:
        # LLM을 사용해 문서와 쿼리 간의 의미적 관련성을 1-10점으로 평가
        prompt = f"""
        아래 주어진 질문과 문서의 관련성을 평가해주세요.

        [평가 기준]
        - 문서가 질문에서 요구하는 정보를 직접적으로 포함하면 8-10점
        - 문서가 질문과 관련된 맥락을 포함하지만 직접적인 답이 아니면 4-7점
        - 문서가 질문과 거의 관련이 없으면 1-3점

        [주의사항]
        - 단순히 비슷한 단어가 등장하는 것은 높은 점수의 근거가 될 수 없습니다
        - 질문의 의도와 문맥을 정확히 파악하여 평가해주세요
        - 시간, 장소, 수치 등 구체적인 정보의 일치 여부를 중요하게 고려해주세요

        질문: {query}
        문서: {content}

        응답은 반드시 다음 JSON 형식이어야 합니다. 백틱은 쓰지마십시오.:
        {{"relevance_score": float}}
        """

        try:
            # LLM에 프롬프트를 전송하고 JSON 형식의 응답을 받음
            response = self.llm.invoke(prompt)
            # 응답에서 relevance_score 값을 추출
            score = json.loads(response.content)["relevance_score"]
            # 점수를 float로 변환하여 반환
            return float(score)
        except Exception as e:
            print(f"Error occurred: {str(e)}")
            return 5.0  # 에러 발생시 중간 점수로 처리하여 시스템 안정성 유지

    def postprocess_documents(self, documents: List[Document], query: str) -> List[Document]:
        # 벡터 검색으로 찾은 4개 문서를 LLM으로 재평가하여 최적의 2개 선택
        print('\n=== LLM이 4개의 검색 결과에 대해서 관련성을 평가합니다. ===')
        scored_docs = []
        for doc in documents:
            # 현재 처리 중인 문서에서 순수 텍스트 컨텐츠만 추출
            content = doc.page_content
            # LLM으로 문서 관련성 점수 계산 (1-10 사이 점수)
            score = self.evaluate_document(query, content)
            # 디버깅/모니터링을 위해 각 문서의 내용과 점수를 출력
            print(f"\nLLM 기반의 평가:\n{content}\n=> 점수: {score}\n")
            # 현재 문서와 계산된 점수를 튜플로 저장
            scored_docs.append((doc, score))

        # 모든 문서를 점수 기준 내림차순으로 정렬하고 상위 2개만 선택하여 반환
        ranked_docs = sorted(scored_docs, key=lambda x: x[1], reverse=True)
        return [doc for doc, _ in ranked_docs[:2]]

In [ ]:
class SemanticRanker:
    # 벡터 검색 결과에 LLM 기반 의미적 평가를 적용하여 최적의 문서를 선별하는 시스템

    def __init__(self, vector_store, scorer):
        # 생성자에서 벡터 검색용 저장소와 LLM 기반 문서 평가기 인스턴스를 받아 저장
        self.vector_store = vector_store  # 벡터 검색용 저장소
        self.scorer = scorer  # LLM 기반 문서 평가기

    def retrieve(self, query: str) -> List[Document]:
        # 벡터 검색으로 유사도 기반 후보 문서 4개를 추출하고 LLM으로 재평가
        vector_results = self.vector_store.similarity_search(query, k=4)

        # 초기 벡터 검색 결과를 디버깅/분석용으로 출력
        print("\n=== 실제 검색 결과 (Top 4) ===")
        for i, doc in enumerate(vector_results, 1):
            print(f"\n검색 문서 {i}:")
            print(doc.page_content)

        # LLM으로 문서들을 재평가하고 재정렬하여 최적의 2개 선택
        reranked_results = self.scorer.postprocess_documents(vector_results, query)

        # 최종 선별된 문서를 디버깅/분석용으로 출력
        print("\n=== LLM의 리랭킹 결과 (Top 2) ===")
        for i, doc in enumerate(reranked_results, 1):
            print(f"\n검색 문서 {i}:")
            print(doc.page_content)

        return reranked_results

In [ ]:
# 문서 평가 및 검색 시스템 선언(초기화)
scorer = DocumentScorer(llm)  # LLM 기반 문서 평가기 생성
ranker = SemanticRanker(vector_store, scorer)  # 벡터 검색과 LLM 평가를 결합한 시스템 생성

In [ ]:
# 최종 답변 생성 함수
def generate_final_answer(query: str, documents: List[Document]) -> str:
    context = "\n\n".join([doc.page_content for doc in documents])

    prompt = f"""다음 검색 결과를 바탕으로 질문에 답변해주세요.
    검색 결과의 정보를 최대한 사용하고, 없는 정보는 답변하지 마세요.

    검색 결과:
    {context}

    질문: {query}

    답변:"""

    response = llm.invoke(prompt)
    return response.content

# 실제 쿼리 실행
query = "19년 말 평양시 소재 기업소에서 달마다 배급받은 음식"
print(f"\n질문: {query}")

# 리랭킹을 통해 최적의 문서 2개 선택
best_documents = ranker.retrieve(query)

# 선택된 문서로 최종 답변 생성
final_answer = generate_final_answer(query, best_documents)
print(f"\n최종 답: {final_answer}")


질문: 19년 말 평양시 소재 기업소에서 달마다 배급받은 음식

=== 실제 검색 결과 (Top 4) ===

검색 문서 1:
화 또는 쌀이나 기름 등 현물로 지급하였다고 한다. 2019년 평양
의 외화벌이 사업소에서는 보수 50달러를 월 2회로 나누어 현금으
로 지급하였다고 하는 사례가 있었고, 평양 외화벌이 식당에서는 매

검색 문서 2:
화 또는 쌀이나 기름 등 현물로 지급하였다고 한다. 2019년 평양
의 외화벌이 사업소에서는 보수 50달러를 월 2회로 나누어 현금으
로 지급하였다고 하는 사례가 있었고, 평양 외화벌이 식당에서는 매

검색 문서 3:
화 또는 쌀이나 기름 등 현물로 지급하였다고 한다. 2019년 평양
의 외화벌이 사업소에서는 보수 50달러를 월 2회로 나누어 현금으
로 지급하였다고 하는 사례가 있었고, 평양 외화벌이 식당에서는 매

검색 문서 4:
화 또는 쌀이나 기름 등 현물로 지급하였다고 한다. 2019년 평양
의 외화벌이 사업소에서는 보수 50달러를 월 2회로 나누어 현금으
로 지급하였다고 하는 사례가 있었고, 평양 외화벌이 식당에서는 매

=== LLM이 4개의 검색 결과에 대해서 관련성을 평가합니다. ===

LLM 기반의 평가:
화 또는 쌀이나 기름 등 현물로 지급하였다고 한다. 2019년 평양
의 외화벌이 사업소에서는 보수 50달러를 월 2회로 나누어 현금으
로 지급하였다고 하는 사례가 있었고, 평양 외화벌이 식당에서는 매
=> 점수: 2.0


LLM 기반의 평가:
화 또는 쌀이나 기름 등 현물로 지급하였다고 한다. 2019년 평양
의 외화벌이 사업소에서는 보수 50달러를 월 2회로 나누어 현금으
로 지급하였다고 하는 사례가 있었고, 평양 외화벌이 식당에서는 매
=> 점수: 3.0


LLM 기반의 평가:
화 또는 쌀이나 기름 등 현물로 지급하였다고 한다. 2019년 평양
의 외화벌이 사업소에서는 보수 50달러를 월 2회로 나누어 현금으
로 지급하였다고 하는 사례가 있었고, 평양 외화벌이 식당에서는 매
=> 점수: 3.0